# 03 — Exportar o Qwen refinado para GGUF e testar com llama.cpp

Este notebook recebe o modelo merged produzido no caderno 02 e cria:

- GGUF em F16 (intermediário);
- GGUF quantizado em **Q4_K_M** para uso local;
- teste de inferência com `llama-cli`.

O `llama.cpp` suporta Qwen3 e o repositório oficial do Qwen também publica GGUFs do Qwen3-0.6B. Aqui fazemos a conversão do **seu** modelo refinado.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

MERGED_DIR = Path('artifacts/qwen3-0.6b-raciocinio-merged')
GGUF_DIR = Path('artifacts/gguf')
LLAMA_DIR = Path('tools/llama.cpp')
GGUF_DIR.mkdir(parents=True, exist_ok=True)
LLAMA_DIR.parent.mkdir(parents=True, exist_ok=True)

assert MERGED_DIR.exists(), 'Execute primeiro o notebook 02 para gerar o modelo merged.'

## 1. Baixar e compilar llama.cpp

Se o laboratório já tiver `llama.cpp`, ajuste `LLAMA_DIR` e pule o clone.

In [ ]:
import subprocess

if not LLAMA_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp', str(LLAMA_DIR)], check=True)
else:
    print('llama.cpp já existe:', LLAMA_DIR)

In [ ]:
# Dependências Python do conversor.
%pip install -U -r tools/llama.cpp/requirements.txt

In [ ]:
# Compilar ferramentas CPU. Para CUDA, use -DGGML_CUDA=ON se seu ambiente tiver toolkit compatível.
subprocess.run(['cmake', '-S', str(LLAMA_DIR), '-B', str(LLAMA_DIR/'build'), '-DCMAKE_BUILD_TYPE=Release'], check=True)
subprocess.run(['cmake', '--build', str(LLAMA_DIR/'build'), '--config', 'Release', '-j'], check=True)

## 2. Converter HF → GGUF F16

In [ ]:
F16 = GGUF_DIR / 'qwen3-0.6b-raciocinio-f16.gguf'
converter = LLAMA_DIR / 'convert_hf_to_gguf.py'

subprocess.run([
    sys.executable, str(converter), str(MERGED_DIR),
    '--outfile', str(F16), '--outtype', 'f16'
], check=True)
print(F16, f'{F16.stat().st_size/1024**3:.2f} GiB')

## 3. Quantizar para Q4_K_M

Q4_K_M costuma ser um bom ponto de partida para laboratório local. Também vale comparar Q8_0 para medir o impacto da quantização.

In [ ]:
def find_binary(name):
    candidates = [
        LLAMA_DIR/'build'/'bin'/name,
        LLAMA_DIR/'build'/'bin'/f'{name}.exe',
        LLAMA_DIR/'build'/name,
        LLAMA_DIR/'build'/f'{name}.exe',
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f'Não encontrei {name}. Candidatos: {candidates}')

quantize = find_binary('llama-quantize')
cli = find_binary('llama-cli')
print('quantize:', quantize)
print('cli:', cli)

In [ ]:
Q4 = GGUF_DIR / 'qwen3-0.6b-raciocinio-q4_k_m.gguf'
subprocess.run([str(quantize), str(F16), str(Q4), 'Q4_K_M'], check=True)
print(Q4, f'{Q4.stat().st_size/1024**2:.1f} MiB')

## 4. Inferência local

No Qwen3, `/think` e `/no_think` são switches de texto úteis no `llama.cpp`. Para modo thinking, use amostragem; evite greedy.

In [ ]:
prompt = '/think Resolva: uma equipe fecha 18 chamados por hora. Quantos chamados fecha em 7 horas?'
cmd = [
    str(cli), '-m', str(Q4),
    '-cnv', '-p', prompt,
    '-n', '256',
    '--temp', '0.6', '--top-p', '0.95', '--top-k', '20',
]
print(' '.join(cmd))
subprocess.run(cmd, check=False)

## 5. Opcional: produzir Q8_0 para comparação

In [ ]:
Q8 = GGUF_DIR / 'qwen3-0.6b-raciocinio-q8_0.gguf'
# Descomente se quiser comparar tamanho/qualidade.
# subprocess.run([str(quantize), str(F16), str(Q8), 'Q8_0'], check=True)
# print(Q8, f'{Q8.stat().st_size/1024**2:.1f} MiB')

## O que medir no laboratório

Monte um conjunto fixo de 20–100 prompts e compare:

- modelo base GGUF vs modelo refinado GGUF;
- respostas com `/think` vs `/no_think`;
- F16 vs Q8_0 vs Q4_K_M;
- acurácia, taxa de formato correto, tokens/s e memória residente.

A quantização não substitui a avaliação: a pergunta relevante é se a perda de qualidade é aceitável **no seu conjunto de tarefas**.